MULTI-SEED DEMO v2 (protokol C, diskusi Claude-Codex 2026-11-22 `langkah-sesudah-hasil-negatif`).

Resep `m2m_base` dari `final_3lv_v2` **identik**, hanya seed penerjemah yang diganti: 42 (reproduksi), 43, 44.
Yang dikunci untuk semua seed:
- juri pemilih checkpoint: satu set weights DistilBERT, dilatih SEKALI dengan kode v2 yang sama (seed 42);
- reranker: weights RoBERTa demo yang tersimpan di output v2 (`model/reranker`), dimuat, tidak dilatih ulang;
- decoder: beam 8 + rerank lambda = 2 (tanpa grid);
- pemilihan checkpoint: skor valid greedy = chrF++ + 30 x ketepatan juri kernel, epoch pertama bila seri.

Fase valid untuk ketiga seed dijalankan dulu. Fase test hanya dijalankan bila seed 42 mereproduksi valid chrF++ 62,66
dan Krama Alus 68,83 (dua desimal). Tidak ada pemilihan antar-seed: model demo tetap seed 42 apa pun hasilnya.
Ketepatan tingkat untuk laporan dinilai lokal dengan juri tetap (`scripts/juri_tetap.py`).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu", "sentencepiece"], check=False)

import glob, json, time, random, re, os, hashlib, gc, shutil
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np, pandas as pd, torch, sacrebleu, transformers
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

REPO, SL, TL = "facebook/m2m100_418M", "id", "jv"
SEEDS = [42, 43, 44]
EPOCHS, OS, BS, LR, WARMUP, JUDGE_SEED, MAXLEN, CTRL_W = 15, 3, 16, 5e-5, 0.1, 42, 64, 30.0
NBEST, LAM = 8, 2.0
TARGET = {"all": 62.66, "krama_alus": 68.83}
OUT = "/kaggle/working"
dev = "cuda"
LEVELS = ["ngoko", "krama", "krama_alus"]
LID = {l: i for i, l in enumerate(LEVELS)}
TAG = {l: f"<{l}>" for l in LEVELS}
INIT_WORDS = {"ngoko": "kowe aku mangan lunga turu", "krama": "sampeyan kula nedha kesah tilem",
              "krama_alus": "panjenenganipun dalem dhahar tindak sare"}
T0 = time.time()
log = lambda *a: print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


find = lambda n: sorted(p for p in glob.glob(f"/kaggle/input/**/{n}.csv", recursive=True) if "/output" not in p)[0]
tr_raw = pd.read_csv(find("train"))
def _load3(n):
    d = pd.read_csv(find(n))
    d = d[d["level"].isin(LEVELS)].copy()
    d["label"] = d["level"].map(LID)
    return d.reset_index(drop=True)
tr, va, te = (_load3(n) for n in ["train", "valid", "test"])
assert (len(tr), len(va), len(te)) == (1395, 175, 176), (len(tr), len(va), len(te))
RR_DIR = os.path.dirname(glob.glob("/kaggle/input/**/model/reranker/config.json", recursive=True)[0])
log("data", len(tr), len(va), len(te), torch.cuda.get_device_name(0), "| reranker", RR_DIR)
tag_src = lambda ind, level: f"{TAG[level]} {ind}"
def bleu(h, r): return sacrebleu.corpus_bleu(h, [r]).score
def chrf(h, r): return sacrebleu.corpus_chrf(h, [r], word_order=2).score
_rr_w = glob.glob(f"{RR_DIR}/*.safetensors") + glob.glob(f"{RR_DIR}/*.bin")
MANIFEST = {"protokol": "C multi-seed (diskusi 2026-11-22-langkah-sesudah-hasil-negatif)", "seeds": SEEDS,
            "hparams": {"epochs": EPOCHS, "os": OS, "bs": BS, "lr": LR, "warmup": WARMUP, "maxlen": MAXLEN,
                        "ctrl_w": CTRL_W, "nbest": NBEST, "lambda": LAM, "judge_seed": JUDGE_SEED},
            "versions": {"python": sys.version.split()[0], "torch": torch.__version__,
                         "transformers": transformers.__version__, "sacrebleu": sacrebleu.__version__},
            "gpu": torch.cuda.get_device_name(0),
            "inputs": {n: sha(find(n)) for n in ["train", "valid", "test"]},
            "reranker_sha256": sha(_rr_w[0]) if _rr_w else None,
            "runs": {}}

In [ ]:
# ---------------------------------------------------- juri kernel (kode v2 persis) + reranker demo (dimuat)
def train_classifier(base, seed):
    ctok = AutoTokenizer.from_pretrained(base)
    cmod = AutoModelForSequenceClassification.from_pretrained(base, num_labels=len(LEVELS)).to(dev)
    seed_all(seed)
    opt = torch.optim.AdamW(cmod.parameters(), lr=5e-5)
    X, y = tr["javanese"].tolist(), tr["label"].tolist()
    for _ in range(4):
        cmod.train(); order = list(range(len(X))); random.shuffle(order)
        for i in range(0, len(order), 32):
            b = order[i:i + 32]
            enc = ctok([X[j] for j in b], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            loss = cmod(**enc, labels=torch.tensor([y[j] for j in b], device=dev)).loss
            opt.zero_grad(); loss.backward(); opt.step()
    cmod.eval()
    return ctok, cmod


def make_lp(ctok, cmod):
    def logprobs(sents):
        out = []
        for i in range(0, len(sents), 64):
            enc = ctok(sents[i:i + 64], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            with torch.no_grad():
                out.append(F.log_softmax(cmod(**enc).logits.float(), -1).cpu())
        return torch.cat(out).numpy()
    return logprobs


def clf_info(lp):
    info = {}
    for n, d in [("valid", va), ("test", te)]:
        p = lp(d["javanese"].tolist()).argmax(-1); y_ = d["label"].values
        info[f"acc_{n}"] = round(float((p == y_).mean()), 4)
        info[f"per_class_acc_{n}"] = {l: round(float((p[y_ == i] == i).mean()), 4) for i, l in enumerate(LEVELS)}
    return info


judge_lp = make_lp(*train_classifier("w11wo/javanese-distilbert-small", JUDGE_SEED))
classify = lambda s: judge_lp(s).argmax(-1).tolist()
rr_tok = AutoTokenizer.from_pretrained(RR_DIR)
rr_mod = AutoModelForSequenceClassification.from_pretrained(RR_DIR).to(dev).float().eval()
rr_lp = make_lp(rr_tok, rr_mod)
# v2: juri acc_valid 0.96 (KA 0.8148), test 0.9375; reranker acc_valid 0.9657, test 0.9261
MANIFEST["judge"] = clf_info(judge_lp); MANIFEST["reranker"] = clf_info(rr_lp)
log("juri", MANIFEST["judge"]); log("reranker", MANIFEST["reranker"])

In [ ]:
# ---------------------------------------------------- MT (fungsi v2 persis)
def load():
    tok = AutoTokenizer.from_pretrained(REPO, src_lang=SL, tgt_lang=TL)
    model = AutoModelForSeq2SeqLM.from_pretrained(REPO)
    tok.add_tokens(list(TAG.values()), special_tokens=True)
    model.resize_token_embeddings(len(tok))
    emb = model.get_input_embeddings().weight
    with torch.no_grad():
        for l, words in INIT_WORDS.items():
            ids = tok(words, add_special_tokens=False)["input_ids"]
            emb[tok.convert_tokens_to_ids(TAG[l])] = emb[ids].mean(0)
    bos = tok.get_lang_id(TL)
    return tok, model.to(dev), bos


def translate(tok, model, bos, srcs, beams=4, bs=48):
    model.eval(); out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=bos, num_beams=beams, max_new_tokens=MAXLEN)
        for j, t in zip(idx, tok.batch_decode(g, skip_special_tokens=True)):
            out[j] = t
    return out


def seq_scores(tok, model, srcs, cands, chunk=16):
    out = []
    for i in range(0, len(srcs), chunk):
        enc = tok(srcs[i:i + chunk], text_target=cands[i:i + chunk], return_tensors="pt",
                  padding=True, truncation=True, max_length=MAXLEN).to(dev)
        labels = enc.pop("labels"); labels[labels == tok.pad_token_id] = -100
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc, labels=labels).logits
        nll = F.cross_entropy(logits.float().transpose(1, 2), labels, ignore_index=-100, reduction="none")
        m = (labels != -100).float()
        out += (-(nll * m).sum(1) / m.sum(1).clamp(min=1)).cpu().tolist()
        del logits, nll
    return np.array(out)


def nbest(tok, model, bos, srcs, k=NBEST, bs=8):
    model.eval(); out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=bos, num_beams=k, num_return_sequences=k, max_new_tokens=MAXLEN)
        txt = tok.batch_decode(g, skip_special_tokens=True)
        sc = seq_scores(tok, model, [srcs[j] for j in idx for _ in range(k)], txt)
        for n, j in enumerate(idx):
            out[j] = (txt[n * k:(n + 1) * k], sc[n * k:(n + 1) * k])
    torch.cuda.empty_cache()
    return out


def rerank(nb, levels, lam=LAM):
    flat = [t for c, _ in nb for t in c]
    lp = rr_lp(flat); res = []; p = 0
    for (c, s), l in zip(nb, levels):
        k = len(c); tot = s + lam * lp[p:p + k, LID[l]]; p += k
        res.append(c[int(np.argmax(tot))])
    return res


def ctrl_acc(hyps, levels):
    p = np.array(classify(hyps)); y = np.array([LID[l] for l in levels])
    return float((p == y).mean()), {l: round(float((p[y == i] == i).mean()), 4) for i, l in enumerate(LEVELS)}


def valid_score(h):
    c = chrf(h, va["javanese"].tolist()); a, alv = ctrl_acc(h, va["level"].tolist())
    return c + CTRL_W * a, round(c, 2), round(a, 4), alv


def per_level(h, d):
    res = {}
    for lv in LEVELS + ["all"]:
        m = (d["level"] == lv) if lv != "all" else d["level"].notna()
        hh, r = [x for x, k in zip(h, m) if k], d.loc[m, "javanese"].tolist()
        res[lv] = {"n": len(r), "chrF++": round(chrf(hh, r), 2), "BLEU": round(bleu(hh, r), 2)}
    return res


def honor_pairs():
    pairs = []
    for _, g in tr_raw.groupby("group"):
        jv = dict(zip(g["level"], g["javanese"]))
        if "krama" in jv and "krama_alus" in jv and jv["krama"] != jv["krama_alus"]:
            pairs.append((f"{TAG['krama_alus']} {jv['krama']}", jv["krama_alus"]))
    return pairs

In [ ]:
# ---------------------------------------------------- FASE VALID untuk ketiga seed
def train_seed(seed):
    seed_all(seed)
    tok, model, bos = load()
    base = [(tag_src(s, l), t, l) for s, l, t in zip(tr["indonesian"], tr["level"], tr["javanese"])]
    id_pairs = [(s, t) for s, t, _ in base] + [(s, t) for s, t, l in base if l == "krama_alus"] * (OS - 1)
    jv_pairs = honor_pairs()

    def make_batches():
        a, b = id_pairs[:], jv_pairs[:]
        random.shuffle(a); random.shuffle(b)
        bt = [(SL, a[i:i + BS]) for i in range(0, len(a), BS)] + [(TL, b[i:i + BS]) for i in range(0, len(b), BS)]
        random.shuffle(bt); return bt

    n_batches = len(make_batches())
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * n_batches
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    best, best_state, hist = -1, None, []
    va_src = [tag_src(s, l) for s, l in zip(va["indonesian"], va["level"])]
    for ep in range(EPOCHS):
        model.train(); tot = n = 0
        for src_lang, b in make_batches():
            tok.src_lang = src_lang
            enc = tok([s for s, _ in b], text_target=[t for _, t in b], return_tensors="pt",
                      padding=True, truncation=True, max_length=MAXLEN).to(dev)
            labels = enc.pop("labels"); labels[labels == tok.pad_token_id] = -100
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**enc, labels=labels).logits
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), labels.view(-1),
                                   ignore_index=-100, label_smoothing=0.0)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item() * len(b); n += len(b)
        tok.src_lang = SL
        sc, c, a, alv = valid_score(translate(tok, model, bos, va_src, 1))
        hist.append({"epoch": ep + 1, "train_loss": round(tot / n, 4), "valid_chrF++": c, "valid_ctrl": a,
                     "valid_ctrl_lv": alv, "score": round(sc, 2)})
        log(f"seed{seed}", hist[-1])
        if sc > best:
            best = sc; best_state = {k: x.detach().to("cpu", copy=True) for k, x in model.state_dict().items()}
    model.load_state_dict(best_state); tok.src_lang = SL
    del best_state, opt, scaler, sched; gc.collect(); torch.cuda.empty_cache()

    hyp = rerank(nbest(tok, model, bos, va_src), va["level"].tolist())
    sc, c, a, alv = valid_score(hyp)
    pv = va.copy(); pv["hyp"] = hyp; pv["juri_kernel"] = [LEVELS[q] for q in classify(hyp)]
    pv.to_csv(f"{OUT}/valid_seed{seed}.csv", index=False)
    res = {"n_pairs": [len(id_pairs), len(jv_pairs)], "history": hist,
           "best_epoch": max(hist, key=lambda h: h["score"])["epoch"],
           "valid_rerank": {"score": round(sc, 2), "chrF++": c, "ctrl": a, "ctrl_lv": alv, "per_level": per_level(hyp, va)},
           "valid_csv_sha256": sha(f"{OUT}/valid_seed{seed}.csv")}
    mdir = f"{OUT}/model_seed{seed}"
    model.half().save_pretrained(mdir); tok.save_pretrained(mdir)
    del model; gc.collect(); torch.cuda.empty_cache()
    log(f"seed{seed} VALID", res["best_epoch"], res["valid_rerank"])
    return res


for s in SEEDS:
    t = time.time()
    MANIFEST["runs"][f"seed{s}"] = train_seed(s)
    MANIFEST["runs"][f"seed{s}"]["detik_valid"] = round(time.time() - t)
    json.dump(MANIFEST, open(f"{OUT}/manifest.json", "w"), indent=1)

In [ ]:
# ---------------------------------------------------- GERBANG reproduksi seed 42 (sebelum test disentuh)
v42 = MANIFEST["runs"]["seed42"]["valid_rerank"]["per_level"]
got = {"all": v42["all"]["chrF++"], "krama_alus": v42["krama_alus"]["chrF++"]}
lengkap = all(f"seed{s}" in MANIFEST["runs"] and os.path.exists(f"{OUT}/model_seed{s}/config.json") for s in SEEDS)
MANIFEST["gerbang"] = {"target": TARGET, "seed42": got, "best_epoch_seed42": MANIFEST["runs"]["seed42"]["best_epoch"],
                       "best_epoch_v2": 14, "lengkap": lengkap,
                       "ok": bool(lengkap and all(round(got[k], 2) == TARGET[k] for k in TARGET))}
log("GERBANG", MANIFEST["gerbang"])
json.dump(MANIFEST, open(f"{OUT}/manifest.json", "w"), indent=1)

In [ ]:
# ---------------------------------------------------- FASE TEST (hanya bila gerbang lolos)
if MANIFEST["gerbang"]["ok"]:
    te_src = [tag_src(s, l) for s, l in zip(te["indonesian"], te["level"])]
    u = sorted(te["indonesian"].unique()); req = [(s, l) for s in u for l in LEVELS]
    c_src = [tag_src(s, l) for s, l in req]
    assert len(te_src) == 176 and len(req) == 225, (len(te_src), len(req))
    for s in SEEDS:
        mdir = f"{OUT}/model_seed{s}"
        tok = AutoTokenizer.from_pretrained(mdir, src_lang=SL, tgt_lang=TL)
        model = AutoModelForSeq2SeqLM.from_pretrained(mdir, torch_dtype=torch.float32).to(dev)
        bos = tok.get_lang_id(TL)
        hyp = rerank(nbest(tok, model, bos, te_src), te["level"].tolist())
        ctl = rerank(nbest(tok, model, bos, c_src), [l for _, l in req])
        pt = te.copy(); pt["hyp"] = hyp; pt.to_csv(f"{OUT}/pred_seed{s}.csv", index=False)
        p = classify(ctl)
        pd.DataFrame({"indonesian": [x for x, _ in req], "requested": [l for _, l in req], "output": ctl,
                      "classified": [LEVELS[q] for q in p]}).to_csv(f"{OUT}/control_seed{s}.csv", index=False)
        a, alv = ctrl_acc(ctl, [l for _, l in req])
        MANIFEST["runs"][f"seed{s}"]["test"] = {"per_level": per_level(hyp, te), "control_juri_kernel": round(a, 4),
                                                 "control_juri_kernel_lv": alv,
                                                 "pred_sha256": sha(f"{OUT}/pred_seed{s}.csv"),
                                                 "control_sha256": sha(f"{OUT}/control_seed{s}.csv")}
        log(f"seed{s} TEST", MANIFEST["runs"][f"seed{s}"]["test"]["per_level"]["all"], "ctrl", round(a, 4))
        del model; gc.collect(); torch.cuda.empty_cache()
    rows = {s: MANIFEST["runs"][f"seed{s}"]["test"]["per_level"] for s in SEEDS}
    agg = {}
    for lv in LEVELS + ["all"]:
        for m in ["chrF++", "BLEU"]:
            v = [rows[s][lv][m] for s in SEEDS]
            agg[f"{lv}_{m}"] = {"rerata": round(float(np.mean(v)), 2), "min": min(v), "maks": max(v)}
    MANIFEST["test_agregat"] = agg
    log("AGREGAT TEST", agg["all_chrF++"], agg["all_BLEU"])
else:
    log("GERBANG GAGAL: fase test dilewati; angka demo tetap hasil historis satu seed.")
MANIFEST["elapsed_s"] = round(time.time() - T0)
for s in SEEDS:   # weights tidak diperlukan sesudah decode; jangan simpan 3 x 0,9 GB di output
    shutil.rmtree(f"{OUT}/model_seed{s}", ignore_errors=True)
json.dump(MANIFEST, open(f"{OUT}/manifest.json", "w"), indent=1)
log("SELESAI"); print(json.dumps({k: v for k, v in MANIFEST.items() if k != "runs"}, indent=1))